# Agentes, Multiagentes y Deep Agents

### UTN-FRSF · Ingeniería en Sistemas de Información · Inteligencia Artificial
**Dr. Jorge Roa · Dra. María de los Milagros Gutiérrez**

---

*¿Cómo pasa un LLM de **responder** a **actuar** de forma autónoma?*

## Recap Clases 1-3 → conexión con hoy

```
  Clase 1            Clase 2                 Clase 3              HOY
  ───────            ───────                 ───────             ─────
  Texto →            Embeddings →            LLM + retrieval     El LLM deja de
  Tokens →           Transformer →           sobre corpus        ser una función
  Embeddings         Texto generado          propio (RAG)        texto→texto
       │                   │                     │                   │
  representar         generar                 dar conocimiento    ACTUAR:
  el texto            el texto                 propio             usar herramientas,
                                                                  planear, iterar
```

Hasta hoy el LLM **respondía**. Hoy le damos **manos**: herramientas, memoria y un
**ciclo** para perseguir un objetivo. El RAG de la Clase 3 va a ser **una herramienta más** del agente.

## Recorrido de la clase

| Bloque | Tema |
|---|---|
| **B1** | ¿Qué es un agente? — LLM puro vs agente, el loop, anatomía |
| **B2** | Patrón **ReAct** + *tool calling* → **Práctica 1** |
| **B3** | Herramientas y **MCP** (Model Context Protocol) |
| **B4** | Sistemas **multiagente** + patrones → **Práctica 2** |
| **B5** | **Frameworks vs Deep-agent harnesses** + estado del arte 2026 → **Práctica 3** |

Mezclamos **concepto + demos en vivo**. Las tres prácticas son notebooks de Colab
que corren con **Groq** (u **Ollama** local).

## ⚙️ Setup

**Las tres prácticas son notebooks separados** que corren en Google Colab. Cada uno
se autoinstala dependencias y configura las credenciales al inicio.

**Dependencias** (cada notebook hace su propio `pip install`):

```bash
pip install langgraph langchain langchain-groq deepagents groq tavily-python python-dotenv
```

**Pre-requisitos:**
- Cuenta gratuita en **Groq** → `GROQ_API_KEY` (Colab Secrets o `.env`).
- *(Opcional)* **Tavily** para búsqueda web real → `TAVILY_API_KEY`. Sin ella, el
  notebook usa un **mock** y corre igual.
- *(Alternativa local)* **Ollama** con `qwen3:8b` / `qwen3:4b`.

# Bloque 1
## ¿Qué es un agente?

---

*Un LLM que no solo habla: percibe, decide, hace y vuelve a mirar.*

## LLM puro vs agente

| | **LLM puro** | **Agente LLM** |
|---|---|---|
| **Entrada → salida** | prompt → texto, una vez | objetivo → secuencia de acciones |
| **Herramientas** | ❌ solo genera tokens | ✅ ejecuta tools (APIs, código, búsqueda, RAG) |
| **Memoria** | ❌ solo el context window | ✅ corto y largo plazo, entre pasos y sesiones |
| **Control de flujo** | el usuario encadena a mano | el modelo **decide** el próximo paso |
| **Iteración** | one-shot | **loop** hasta cumplir el objetivo |

> Un agente = **LLM + herramientas + memoria + un loop que el propio modelo dirige.**
> No es un modelo más grande; es el mismo modelo dentro de una *arquitectura*.

## El loop del agente

El corazón de todo agente es un ciclo: **percibir → razonar → actuar → observar**, y repetir.

<img src="figures/agent_loop.svg" alt="Ciclo del agente: percibir, razonar, actuar, observar" class="slide-figure" style="width: 78%;"/>

> *Nota:* este es el ciclo **reactivo** — se cierra sobre el **resultado de las acciones**. En un ambiente **dinámico** falta además **re-percibir el entorno** por cambios exógenos (otros agentes, el tiempo, eventos que llegan) que no causaron las tools.


## Anatomía de un agente

<img src="figures/agent_anatomy.svg" alt="Componentes de un agente: LLM, memoria, herramientas, planificador, loop de reflexión, control" class="slide-figure" style="width: 88%;"/>

El **LLM es el cerebro** que razona y decide; el resto son los órganos que le dan
percepción, memoria y manos. El módulo de **control** evita acciones peligrosas.

## De chatbot a agente — un ejemplo concreto

**Pregunta:** *"¿Cuántos artículos de la ley provincial 21.526 mencionan 'depósito',
y cuánto es el 12% del total?"*

```
  Chatbot (LLM puro)              Agente
  ─────────────────              ──────
  Inventa un número.    │  Thought: necesito buscar en la ley → uso el RAG.
  No tiene la ley ni    │  Action: rag_buscar("depósito")        → 8 artículos
  sabe contar exacto.   │  Thought: ahora calculo 12% de 8.
                        │  Action: calculadora("8 * 0.12")        → 0.96
                        │  Thought: ya tengo todo → respondo con cita.
                        │  Answer: "8 artículos; el 12% es 0,96." [cita arts.]
```

El agente **combina** el RAG de la Clase 3 (como *tool*) + una calculadora + razonamiento.
Cada paso es **verificable y trazable**.

# Bloque 2
## ReAct: razonar + actuar

---

*El patrón que convirtió a los LLMs en agentes: pensar y actuar, intercalados.*

## El patrón ReAct

**ReAct** = **Rea**soning + **Act**ing (Yao et al., 2022). El modelo **intercala**
razonamiento y acción en un bucle explícito:

<img src="figures/react_cycle.svg" alt="Ciclo ReAct: Thought, Action, Observation" class="slide-figure" style="width: 70%;"/>

- **Thought** — el modelo razona en voz alta qué hacer.
- **Action** — invoca una herramienta con argumentos.
- **Observation** — recibe el resultado y lo incorpora al contexto.

El loop se repite hasta que el modelo decide que tiene la respuesta final.

## ReAct en el prompt — una traza real

El truco es **enseñarle el formato** en el system prompt y dejar que el modelo lo siga:

```
Pregunta: ¿Qué temperatura hace en Santa Fe y cuánto es eso en Fahrenheit?

Thought:  Necesito la temperatura actual de Santa Fe.
Action:   clima["Santa Fe"]
Observation: 30 °C

Thought:  Ahora convierto 30 °C a Fahrenheit: (30 × 9/5) + 32.
Action:   calculadora["(30 * 9/5) + 32"]
Observation: 86

Thought:  Ya tengo todo.
Answer:   En Santa Fe hay 30 °C, equivalente a 86 °F.
```

> El LLM **no ejecuta** las tools: genera el texto `Action: ...`; nuestro código lo
> parsea, ejecuta la tool y le devuelve la `Observation`. Eso lo construimos en la Práctica 1.

## Tool calling / function calling

Los modelos modernos traen *tool calling* **nativo**: en vez de parsear texto, el modelo devuelve un **JSON estructurado** con qué función llamar y con qué argumentos.

<img src="figures/tool_calling.svg" alt="Flujo de tool calling: definición de tools, decisión del modelo, ejecución, resultado" class="slide-figure" style="width: 80%;"/>

1. Le pasamos el **catálogo de tools** (nombre + descripción + schema). 2. El modelo decide **si** usar una y devuelve `{"name": ..., "arguments": {...}}`. 3. El **runtime ejecuta** la función. 4. El modelo razona con el resultado y sigue (o responde).


## 🧪 Práctica 1 — ReAct + tools desde cero

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/react_tools_scratch.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Qué vamos a construir:**

1. El loop ReAct **a mano**, en Python puro — parsear `Thought/Action/Observation`,
   ejecutar tools y realimentar al modelo (Groq). *Desmitificar el agente.*
2. Tools simples y autocontenidas: **calculadora** + **búsqueda web** (Tavily, con mock).
3. La **misma idea reescrita con LangGraph** — un `StateGraph` con nodos *agent* y *tools*,
   para ver qué nos da un framework por encima del loop manual.

> *Mensaje:* un agente ReAct cabe en ~30 líneas. El framework agrega estructura, no magia.

# Bloque 3
## Herramientas y MCP

---

*Sin herramientas el agente solo habla; con ellas, actúa. MCP es el enchufe estándar.*

## Tipos de herramientas

| Tipo | Ejemplos | Para qué |
|---|---|---|
| **Búsqueda / Retrieval** | Tavily, web search, **el RAG de Clase 3** | traer conocimiento externo o propio |
| **Cómputo** | calculadora, ejecutar código, sandbox Python | precisión que el LLM no tiene |
| **APIs / acciones** | clima, calendario, mandar mail, crear ticket | actuar sobre el mundo |
| **Archivos / sistema** | leer/escribir archivos, shell, git | trabajar sobre un workspace |
| **Bases de datos** | SQL, vector store | consultar datos estructurados |

> Diseñar buenas tools (nombres claros, descripciones, schemas) es **la mitad** de hacer
> que un agente funcione. El modelo elige en base a lo que vos le describís.

## El problema: integraciones M×N

Cada agente tiene que integrarse con cada herramienta **a mano**. Con *M* agentes y *N*
herramientas → *M×N* integraciones a medida. No escala.

<img src="figures/mcp.svg" alt="MCP: de integraciones M por N a un protocolo estándar cliente-servidor" class="slide-figure" style="width: 86%;"/>

**MCP (Model Context Protocol)** — estándar abierto de Anthropic (2024). Un *protocolo*
común entre **clientes** (el agente/host) y **servidores** (que exponen tools, datos, prompts).

## MCP en breve

- **Servidor MCP**: expone *tools*, *resources* (datos) y *prompts* de forma estándar.
  Lo escribe quien tiene la integración (GitHub, Slack, una base interna, tu RAG…).
- **Cliente MCP**: vive dentro del agente/host (Claude, IDEs, agentes custom) y descubre
  y consume lo que el servidor ofrece — **sin código a medida por cada par**.
- **Por qué importa en 2026**: pasó de *M×N* a *M+N*. Un servidor MCP de tu RAG sirve
  para **cualquier** agente compatible. Es el "USB-C" de las herramientas de agentes.

> No vamos a implementar un servidor MCP hoy, pero es el estándar que vas a encontrar
> apenas conectes un agente a herramientas reales.

# Bloque 4
## Sistemas multiagente

---

*Cuando un solo agente no alcanza: dividir para conquistar.*

## ¿Por qué varios agentes?

Un único agente con 20 herramientas y un prompt gigante se vuelve **frágil**: se confunde,
pierde foco, satura su context window.

| Ventaja | Por qué |
|---|---|
| **Especialización** | cada agente tiene un rol, pocas tools y un prompt afilado |
| **Gestión de contexto** | cada uno usa su propia ventana → no compiten por espacio |
| **Modularidad** | testeás, reemplazás o reusás cada agente por separado |
| **Paralelismo** | subtareas independientes corren a la vez |

> El costo: más latencia, más tokens y **coordinación**. Multiagente no es gratis —
> se justifica cuando la tarea es genuinamente compleja o heterogénea.

## Patrones multiagente

<img src="figures/multiagent_patterns.svg" alt="Patrones multiagente: secuencial, router, orquestador, paralelo, jerárquico" class="slide-figure" style="width: 92%;"/>

- **Secuencial (prompt chaining)** — la salida de uno alimenta al siguiente.
- **Router** — un agente clasifica y deriva al especialista correcto.
- **Orquestador** — un manager planifica, delega y junta resultados.
- **Paralelo** — subtareas independientes en simultáneo, luego se agregan.
- **Jerárquico** — orquestadores que coordinan a otros orquestadores.

## Single-agent vs multiagente — cuándo conviene

| Usá **un solo agente** si… | Usá **multiagente** si… |
|---|---|
| la tarea es acotada y homogénea | hay roles claramente distintos (investigar, redactar, revisar) |
| pocas herramientas | muchas tools que conviene repartir |
| la latencia importa | podés paralelizar subtareas independientes |
| querés simplicidad y bajo costo | la calidad justifica más tokens y coordinación |

> **Regla práctica:** empezá con **un** agente bien diseñado. Pasá a multiagente solo
> cuando el agente único se rompe por complejidad o contexto. *No al revés.*

## A2A — Agent2Agent

**Protocolo abierto** para que distintos agentes **se descubran, se comuniquen y se deleguen tareas** entre sí — sin importar el framework o el vendor. Google (abril 2025) → hoy en la **Linux Foundation**.

| Concepto | Qué es |
|---|---|
| **Agent Card** | un JSON público (`/.well-known/agent.json`) donde el agente declara **quién es y qué sabe hacer** → permite el *descubrimiento* |
| **Client ↔ Remote** | un agente *cliente* le delega trabajo a un agente *remoto* |
| **Task** | la unidad de trabajo, con estados (`submitted → working → completed`) |
| **Messages / Artifacts** | lo que se intercambian y el resultado que se devuelve |

Sobre **HTTP + JSON-RPC + SSE** (streaming para tareas largas).

> Hoy cada framework arma agentes **aislados**; A2A busca volverlos **interoperables** — el "protocolo de equipo" entre agentes.


## MCP + A2A — dos estándares complementarios

Ya vimos los dos: **MCP** conecta al agente con **herramientas**; **A2A** lo conecta con **otros agentes**. Son ejes distintos, no competidores.

<img src="figures/mcp_vs_a2a.svg" alt="MCP conecta el agente con herramientas; A2A conecta el agente con otros agentes" class="slide-figure" style="width: 82%;"/>

> Un mismo agente usa **MCP** para sus tools y **A2A** para trabajar en equipo con otros agentes — aunque sean de distinto framework o vendor.


## 🧪 Práctica 2 — Multiagente con LangGraph

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/multiagente_langgraph.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Qué vamos a construir:** un mini-equipo con patrón **orquestador**:

```
            ┌──────────────┐
   tarea →  │ Orquestador  │  decide y delega
            └──────┬───────┘
        ┌──────────┼──────────┐
        ▼          ▼          ▼
   Investigador  Redactor   Revisor
   (búsqueda)   (escribe)  (critica y corrige)
```

Con **LangGraph**: estado compartido, nodos por agente, y aristas que definen el flujo.
Sobre **Groq** (u Ollama). Reusa las tools de la Práctica 1.

# Bloque 5
## Frameworks vs Deep-agent harnesses

---

*El ecosistema 2026: ¿armo el agente, o uso uno ya armado?*

## Dos niveles de abstracción

Distinción real y útil (Anthropic, *Building Effective Agents*; LangChain, *deep agents*): **¿quién es dueño del loop?**

<img src="figures/taxonomy_frameworks_vs_harness.svg" alt="Taxonomía: frameworks de orquestación vs deep-agent harnesses" class="slide-figure" style="width: 74%;"/>

- **Tier 1 — Frameworks de orquestación:** te dan los ladrillos; **vos** armás el grafo/loop.
- **Tier 2 — Deep-agent harnesses** *(harness = la maquinaria que rodea al LLM)*: ya traen un loop **opinado** (planificación + subagentes + filesystem/memoria); vos lo configurás y embebés.


## Tier 1 — Frameworks de orquestación

Vos definís el control de flujo, las tools y el estado.

| Framework | Idea distintiva |
|---|---|
| **smolagents** (HuggingFace) | minimalista (~1k LOC); agentes que "piensan en código" |
| **LangChain / LangGraph** | LangGraph = grafos de estado explícitos; control fino del flujo |
| **LlamaIndex** | agentes sobre datos/RAG; fuerte en *retrieval* |
| **PydanticAI** | tipado y *structured outputs*; "el FastAPI de los agentes" |
| **CrewAI** | multiagente por **roles** ("crews"); muy popular |
| **Agno** (ex-phidata) | ultraliviano y rápido; "teams" multiagente |
| **OpenAI Agents SDK** | sucesor de *Swarm*; handoffs, guardrails, tracing |
| **Google ADK** | framework code-first de Google; soporta **A2A** |

> Otros: **Microsoft Agent Framework** (sucesor de AutoGen + Semantic Kernel) · **Mastra** (TypeScript). Las **dos prácticas** de hoy son Tier 1 (LangGraph): vos ves y controlás cada nodo.


## Tier 2 — Deep-agent harnesses

Un *harness* "profundo" trae de fábrica lo que en Tier 1 tendrías que cablear vos. Cuatro pilares (LangChain `deepagents`):

1. **System prompt detallado** — le enseña a planificar y verificar antes de actuar.
2. **Planificación** — una tool tipo *TODO list* (`write_todos`) que arma y adapta el plan.
3. **Subagentes** — delega subtareas a agentes con **contexto aislado**.
4. **Filesystem / memoria virtual** — descarga contexto a archivos; memoria de largo plazo.

> Pensados para tareas **largas y multi-paso** (ej: agentes de coding). El harness gestiona
> el contexto por vos — que es **el** problema difícil de los agentes de larga duración.

## Tier 2 — el ecosistema 2026

| Harness | Qué es | Modelo |
|---|---|---|
| **Claude Agent SDK** | el harness que mueve Claude Code, como librería (ex *Claude Code SDK*) | Claude |
| **opencode** | agente de coding open-source, SDK JS/TS + server headless | multi-proveedor (incl. **Groq**) |
| **pi.dev** | harness minimalista (4 tools + extensiones), MIT | multi-proveedor (incl. **Groq/Ollama**) |
| **LangChain `deepagents`** | harness *deep agent* en **Python**, sobre LangGraph | cualquiera vía `init_chat_model` (**Groq/Ollama**) |

```python
# deepagents — el harness arma el loop; vos solo das modelo, tools y prompt
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="groq:llama-3.3-70b-versatile",      # model-agnostic
    tools=[buscar_web, calculadora],
    system_prompt="Sos un asistente de investigación. Planificá antes de actuar.",
)
agent.invoke({"messages": [{"role": "user", "content": "Armá un informe sobre ..."}]})
```

> **opencode** y **pi.dev** son JS/TS / orientados a terminal → los vemos como concepto. Pero **`deepagents` es Python y corre con Groq** → lo probamos en la **Práctica 3**.

## 🧪 Práctica 3 — Deep agent con deepagents

<a href="https://colab.research.google.com/github/jorgeroa/ia-utn-frsf/blob/main/clase04/notebooks/deepagents_harness.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Qué vamos a construir:** el **mismo tipo de tarea** que en la Práctica 1, pero con un **harness Tier 2** — sin escribir el loop ni el planner.

- `create_deep_agent(...)` sobre **Groq**: le damos **modelo + tools + prompt**, nada más.
- El harness aporta de fábrica: **planificación** (`write_todos`), **subagentes** (contexto aislado) y **filesystem virtual**.
- Tarea multi-paso ("investigá y escribí un informe") → se ven el plan, la delegación y los archivos.

> *Contraste:* en la Práctica 1 vos armaste el loop ReAct a mano; acá lo trae el harness. **Mismo agente, otra altura.**


## ¿Cuándo cada tier?

| Elegí **Tier 1** (frameworks) si… | Elegí **Tier 2** (harness) si… |
|---|---|
| querés **control total** del flujo | querés arrancar rápido con un loop probado |
| el agente es a medida / didáctico | la tarea es larga, multi-paso, tipo "coding agent" |
| necesitás entender cada paso | la gestión de contexto/planificación te complica |
| es producción con lógica propia | querés subagentes + filesystem "gratis" |

## Horizonte — estado del arte 2026

- **Agentes autónomos "personales"** — *always-on*, de propósito general: corren solos en tu servidor y persiguen objetivos por su cuenta. **OpenClaw** (construido sobre **pi.dev**) y **Hermes Agent** (Nous Research, con loop de **auto-mejora**) son los dos referentes.
- **Software autónomo** — **OpenHands** (ex-OpenDevin): agentes que escriben código, corren comandos y navegan para resolver tareas de ingeniería de punta a punta.

> La frontera se mueve hacia agentes que corren **solos por horas**. Eso abre la pregunta de siempre: ¿cuánta autonomía le damos, y con qué controles?


## Límites actuales — la otra cara

| Límite | Qué significa en la práctica |
|---|---|
| **Fiabilidad** | un paso malo se propaga; errores compuestos en cadenas largas |
| **Costo** | cada paso es una llamada al LLM; multiagente multiplica tokens |
| **Latencia** | loops + tools = segundos o minutos por tarea |
| **Seguridad** | tools con efectos reales (mail, shell, pagos) → necesitás *human-in-the-loop*, permisos, sandbox |

> Un agente es tan confiable como su **tool peor diseñada** y su **control más débil**.
> En dominios sensibles, la autonomía total todavía **no** es la respuesta.

## Síntesis

- Un **agente** = LLM + herramientas + memoria + un **loop** que el modelo dirige.
- **ReAct** (Thought → Action → Observation) es el patrón base; el *tool calling* nativo lo hace robusto.
- Las **herramientas** le permiten al agente **actuar** (buscar, calcular, llamar APIs, leer/escribir); **MCP** es el estándar para conectarlas.
- **Multiagente** divide para conquistar (solo cuando un agente único se queda corto); **A2A** es el estándar para que esos agentes se comuniquen entre sí.
- Ecosistema: **Tier 1** (armás el loop) vs **Tier 2** (harness que ya lo trae).

> El RAG de la Clase 3 no se reemplaza: se vuelve **una herramienta** dentro del agente.
> Eso es exactamente lo que pide el **TP integrador**.

## Referencias

- Yao et al. (2022). *ReAct: Synergizing Reasoning and Acting in Language Models.* https://arxiv.org/abs/2210.03629
- Anthropic (2024). *Building Effective Agents.* https://www.anthropic.com/research/building-effective-agents
- Wang et al. (2023). *A Survey on LLM-based Autonomous Agents.* https://arxiv.org/abs/2308.11432
- Anthropic (2024). *Model Context Protocol.* https://modelcontextprotocol.io
- LangChain. *Deep Agents.* https://docs.langchain.com/oss/python/deepagents/overview
- LangGraph docs. https://langchain-ai.github.io/langgraph/
- HuggingFace. *smolagents.* https://github.com/huggingface/smolagents
- opencode. https://opencode.ai · pi.dev. https://pi.dev · Nous Research, *Hermes 3.* https://nousresearch.com/hermes3